In [1]:
# Smart Healing: AI-Based Disease Diagnosis System
# Artificial Intelligence (CSC-350) - Spring 2026
# Sukkur IBA University

import numpy as np
import pandas as pd
from collections import deque
import heapq
import random
import math
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    mean_absolute_error, mean_squared_error
)



# 1. CREATE PATIENT DATASET


def create_dataset():
    np.random.seed(42)
    n = 200
    glucose  = np.clip(np.random.normal(120, 30, n), 60, 200).astype(int)
    bmi      = np.clip(np.random.normal(30, 7, n), 15, 55).round(1)
    age      = np.clip(np.random.normal(35, 12, n), 18, 80).astype(int)
    bp       = np.clip(np.random.normal(72, 12, n), 40, 110).astype(int)
    insulin  = np.clip(np.random.normal(80, 50, n), 0, 300).astype(int)
    skin     = np.clip(np.random.normal(25, 10, n), 5, 60).astype(int)
    preg     = np.clip(np.random.randint(0, 10, n), 0, 10)
    pedigree = np.clip(np.random.exponential(0.5, n), 0.05, 2.5).round(3)

    df = pd.DataFrame({
        "Glucose": glucose, "BMI": bmi, "Age": age,
        "BloodPressure": bp, "Insulin": insulin,
        "SkinThickness": skin, "Pregnancies": preg,
        "DiabetesPedigree": pedigree
    })

    score = (
        (df["Glucose"] > 125).astype(int) * 2 +
        (df["BMI"] > 30).astype(int) +
        (df["Age"] > 40).astype(int) +
        (df["DiabetesPedigree"] > 0.5).astype(int)
    )
    df["Disease"] = (score >= 3).astype(int)
    df.to_csv("smart_healing_dataset.csv", index=False) # Back-end log export

    return df


# ──────────────────────────────────────────────
# 2. RATIONAL AGENT (PEAS)
# ──────────────────────────────────────────────

class DiagnosticAgent:
    def __init__(self):
        self.name = "SmartHealingAgent"

    def diagnose(self, patient, model, scaler):
        data = np.array(list(patient.values())).reshape(1, -1)
        data = scaler.transform(data)
        prediction  = model.predict(data)[0]
        probability = model.predict_proba(data)[0][1]

        diagnosis = "Diabetic" if prediction == 1 else "Healthy"
        risk = "HIGH" if probability > 0.7 else "MEDIUM" if probability > 0.4 else "LOW"

        return {
            "Diagnosis"  : diagnosis,
            "Risk Level" : risk,
            "Confidence" : f"{probability * 100:.1f}%"
        }


# ──────────────────────────────────────────────
# 3. SEARCH ALGORITHMS
# ──────────────────────────────────────────────

SYMPTOM_GRAPH = {
    "start":              [("high_glucose", 1), ("fatigue", 2), ("frequent_urination", 1)],
    "high_glucose":       [("diabetes", 3), ("prediabetes", 2)],
    "fatigue":            [("diabetes", 4), ("anemia", 3)],
    "frequent_urination": [("diabetes", 2), ("uti", 1)],
    "prediabetes":        [("diabetes", 1)],
    "diabetes": [], "anemia": [], "uti": [],
}

HEURISTIC = {
    "start": 4, "high_glucose": 2, "fatigue": 3,
    "frequent_urination": 2, "prediabetes": 1,
    "diabetes": 0, "anemia": 5, "uti": 5,
}


def bfs(graph, start, goal):
    queue = deque([[start]])
    visited = set()
    while queue:
        path = queue.popleft()
        node = path[-1]
        if node == goal:
            return path
        if node not in visited:
            visited.add(node)
            for neighbor, _ in graph.get(node, []):
                queue.append(path + [neighbor])
    return None


def dfs(graph, start, goal, path=None, visited=None):
    if path is None:
        path, visited = [start], set()
    if start == goal:
        return path
    visited.add(start)
    for neighbor, _ in graph.get(start, []):
        if neighbor not in visited:
            result = dfs(graph, neighbor, goal, path + [neighbor], visited)
            if result:
                return result
    return None


def ucs(graph, start, goal):
    heap = [(0, [start])]
    visited = set()
    while heap:
        cost, path = heapq.heappop(heap)
        node = path[-1]
        if node == goal:
            return path, cost
        if node not in visited:
            visited.add(node)
            for neighbor, edge_cost in graph.get(node, []):
                heapq.heappush(heap, (cost + edge_cost, path + [neighbor]))
    return None, 0


def astar(graph, heuristic, start, goal):
    heap = [(heuristic[start], 0, [start])]
    visited = set()
    while heap:
        f, g, path = heapq.heappop(heap)
        node = path[-1]
        if node == goal:
            return path, g
        if node not in visited:
            visited.add(node)
            for neighbor, cost in graph.get(node, []):
                new_g = g + cost
                heapq.heappush(heap, (new_g + heuristic.get(neighbor, 0), new_g, path + [neighbor]))
    return None, 0


# ──────────────────────────────────────────────
# 4. GENETIC ALGORITHM (Feature Selection)
# ──────────────────────────────────────────────

def genetic_algorithm(X_train, y_train, X_test, y_test, n_features=8):
    def fitness(chromosome):
        selected = [i for i, b in enumerate(chromosome) if b == 1]
        if not selected:
            return 0.0
        model = KNeighborsClassifier(n_neighbors=5)
        model.fit(X_train[:, selected], y_train)
        return accuracy_score(y_test, model.predict(X_test[:, selected]))

    population = [[random.randint(0, 1) for _ in range(n_features)] for _ in range(10)]
    best, best_score = None, 0.0

    for _ in range(5):
        scored = sorted([(fitness(c), c) for c in population], reverse=True)
        if scored[0][0] > best_score:
            best_score = scored[0][0]
            best = scored[0][1]
        survivors = [c for _, c in scored[:5]]
        children = []
        for i in range(0, len(survivors) - 1, 2):
            point = random.randint(1, n_features - 1)
            children.append(survivors[i][:point] + survivors[i+1][point:])
            children.append(survivors[i+1][:point] + survivors[i][point:])
        for child in children:
            for j in range(n_features):
                if random.random() < 0.2:
                    child[j] = 1 - child[j]
        population = survivors + children

    selected = [i for i, b in enumerate(best) if b == 1]
    return selected, best_score


# ──────────────────────────────────────────────
# 5. MIN-MAX WITH ALPHA-BETA PRUNING
# ──────────────────────────────────────────────

GAME_TREE = {
    "name": "Order Tests?",
    "children": [
        {"name": "Blood Test", "children": [
            {"name": "Glucose Test", "value": 9, "children": []},
            {"name": "HbA1c Test",   "value": 7, "children": []},
        ]},
        {"name": "Skip Tests", "children": [
            {"name": "Use History",  "value": 5, "children": []},
            {"name": "Random Guess", "value": 2, "children": []},
        ]},
    ]
}


def minimax(node, depth, is_max, alpha, beta):
    if depth == 0 or not node.get("children"):
        return node.get("value", 0)
    if is_max:
        best = float('-inf')
        for child in node["children"]:
            best = max(best, minimax(child, depth - 1, False, alpha, beta))
            alpha = max(alpha, best)
            if beta <= alpha:
                break
        return best
    else:
        best = float('inf')
        for child in node["children"]:
            best = min(best, minimax(child, depth - 1, True, alpha, beta))
            beta = min(beta, best)
            if beta <= alpha:
                break
        return best


# ──────────────────────────────────────────────
# 6. MACHINE LEARNING MODELS
# ──────────────────────────────────────────────

def train_models(X_train, y_train):
    knn = KNeighborsClassifier(n_neighbors=7)
    knn.fit(X_train, y_train)

    dt = DecisionTreeClassifier(max_depth=6, random_state=42)
    dt.fit(X_train, y_train)

    lr = LogisticRegression(max_iter=200, random_state=42)
    lr.fit(X_train, y_train)

    return knn, dt, lr


# ──────────────────────────────────────────────
# 7. EVALUATE MODEL
# ──────────────────────────────────────────────

def evaluate(model, X_test, y_test, name):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()

    acc         = accuracy_score(y_test, y_pred) * 100
    precision   = precision_score(y_test, y_pred, zero_division=0) * 100
    sensitivity = recall_score(y_test, y_pred, zero_division=0) * 100
    specificity = (tn / (tn + fp)) * 100 if (tn + fp) > 0 else 0
    f1          = f1_score(y_test, y_pred, zero_division=0)
    geo_mean    = math.sqrt((sensitivity / 100) * (specificity / 100))
    auc         = roc_auc_score(y_test, y_prob)
    mae         = mean_absolute_error(y_test, y_pred)
    mse         = mean_squared_error(y_test, y_pred)

    print(f"\n  Model       : {name}")
    print(f"  Accuracy    : {acc:.1f}%")
    print(f"  Precision   : {precision:.1f}%")
    print(f"  Sensitivity : {sensitivity:.1f}%")
    print(f"  Specificity : {specificity:.1f}%")
    print(f"  F1-Score    : {f1:.4f}")
    print(f"  Geo Mean    : {geo_mean:.4f}")
    print(f"  AUC-ROC     : {auc:.4f}")
    print(f"  MAE         : {mae:.4f}")
    print(f"  MSE         : {mse:.4f}")
    print(f"  Confusion Matrix -> TP={tp} FP={fp} FN={fn} TN={tn}")

    return f1


# ──────────────────────────────────────────────
# 8. MAIN - RUN EVERYTHING
# ──────────────────────────────────────────────

def main():
    print("\n" + "=" * 55)
    print("   SMART HEALING: AI-BASED DISEASE DIAGNOSIS")
    print("   Artificial Intelligence (CSC-350) - Spring 2026")
    print("   Sukkur IBA University")
    print("=" * 55)

    # Dataset
    print("\n--- 1. PATIENT DATASET ---")
    df = create_dataset()
    print(f"  Total Patients : {len(df)}")
    print(f"  Healthy        : {(df['Disease'] == 0).sum()}")
    print(f"  Diabetic       : {(df['Disease'] == 1).sum()}")
    print(f"\n  First 5 rows:")
    print(df.head().to_string(index=False))

    # Rational Agent
    print("\n--- 2. RATIONAL AGENT (PEAS) ---")
    print("  Performance : Accuracy, Sensitivity, F1-Score")
    print("  Environment : Patient records, lab results")
    print("  Actuators   : Diagnosis, Risk Level, Confidence")
    print("  Sensors     : Glucose, BMI, Age, BloodPressure, etc.")

    # Search Algorithms
    print("\n--- 3. SEARCH ALGORITHMS ---")
    goal = "diabetes"
    path = bfs(SYMPTOM_GRAPH, "start", goal)
    print(f"  BFS : {' -> '.join(path)}")
    path = dfs(SYMPTOM_GRAPH, "start", goal)
    print(f"  DFS : {' -> '.join(path)}")
    path, cost = ucs(SYMPTOM_GRAPH, "start", goal)
    print(f"  UCS : {' -> '.join(path)}  (cost={cost})")
    path, cost = astar(SYMPTOM_GRAPH, HEURISTIC, "start", goal)
    print(f"  A*  : {' -> '.join(path)}  (cost={cost})")

    # Data Preparation
    X = df.drop("Disease", axis=1).values
    y = df["Disease"].values
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    scaler     = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc  = scaler.transform(X_test)
    feature_names = list(df.columns[:-1])

    # Genetic Algorithm
    print("\n--- 4. GENETIC ALGORITHM (Feature Selection) ---")
    selected_idx, ga_score = genetic_algorithm(X_train_sc, y_train, X_test_sc, y_test)
    selected = [feature_names[i] for i in selected_idx]
    print(f"  Original : {feature_names}")
    print(f"  Selected : {selected}")
    print(f"  Score    : {ga_score * 100:.2f}%")

    # Min-Max
    print("\n--- 5. MIN-MAX + ALPHA-BETA PRUNING ---")
    best_val = minimax(GAME_TREE, depth=3, is_max=True, alpha=float('-inf'), beta=float('inf'))
    print(f"  Best Score     : {best_val}")
    print(f"  Best Action    : Order Blood Test -> Glucose Test")

    # Train Models
    print("\n--- 6. MACHINE LEARNING MODELS ---")
    knn, dt, lr = train_models(X_train_sc, y_train)
    print("  Trained: kNN, Decision Tree, Logistic Regression")

    # Evaluation
    print("\n--- 7. EVALUATION METRICS ---")
    f1_knn = evaluate(knn, X_test_sc, y_test, "kNN (k=7)")
    f1_dt  = evaluate(dt,  X_test,    y_test, "Decision Tree")
    f1_lr  = evaluate(lr,  X_test_sc, y_test, "Logistic Regression")
    best_name = ["kNN", "Decision Tree", "Logistic Regression"][
        [f1_knn, f1_dt, f1_lr].index(max(f1_knn, f1_dt, f1_lr))
    ]
    print(f"\n  Best Model : {best_name}")

    # K-Fold Cross Validation
    print("\n--- 8. K-FOLD CROSS VALIDATION ---")
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(LogisticRegression(max_iter=200), X_train_sc, y_train, cv=kf)
    print(f"  Mean Accuracy : {scores.mean() * 100:.2f}%")
    print(f"  Std Dev       : ±{scores.std() * 100:.2f}%")
    print(f"  Fold Scores   : {[round(s * 100, 1) for s in scores]}")

    # Feature Importance
    print("\n--- 9. FEATURE IMPORTANCE (Decision Tree) ---")
    ranked = sorted(zip(feature_names, dt.feature_importances_), key=lambda x: x[1], reverse=True)
    for i, (feat, imp) in enumerate(ranked, 1):
        bar = "█" * int(imp * 40)
        print(f"  {i}. {feat:<20} {imp:.4f}  {bar}")

    # K-Means Clustering
    print("\n--- 10. K-MEANS CLUSTERING ---")
    kmeans   = KMeans(n_clusters=2, random_state=42, n_init=10)
    clusters = kmeans.fit_predict(X_train_sc)
    print(f"  Cluster 0 (Low Risk)  : {(clusters == 0).sum()} patients")
    print(f"  Cluster 1 (High Risk) : {(clusters == 1).sum()} patients")

    # Diagnose New Patient
    print("\n--- 11. DIAGNOSE A NEW PATIENT ---")
    agent = DiagnosticAgent()
    new_patient = {
        "Glucose": 155, "BMI": 34.5, "Age": 48,
        "BloodPressure": 80, "Insulin": 120,
        "SkinThickness": 30, "Pregnancies": 3,
        "DiabetesPedigree": 0.8
    }
    print(f"  Input : {new_patient}")
    result = agent.diagnose(new_patient, lr, scaler)
    for key, val in result.items():
        print(f"  {key:<12}: {val}")

    print("\n" + "=" * 55)
    print("  All sections completed successfully.")
    print("=" * 55 + "\n")


if __name__ == "__main__":
    main()


   SMART HEALING: AI-BASED DISEASE DIAGNOSIS
   Artificial Intelligence (CSC-350) - Spring 2026
   Sukkur IBA University

--- 1. PATIENT DATASET ---
  Total Patients : 200
  Healthy        : 131
  Diabetic       : 69

  First 5 rows:
 Glucose  BMI  Age  BloodPressure  Insulin  SkinThickness  Pregnancies  DiabetesPedigree  Disease
     134 32.5   18             81      126             38            7             0.872        1
     115 33.9   27             60       54             34            9             0.549        0
     139 37.6   35             82       84             25            6             0.065        1
     165 37.4   35             88       56             18            5             0.325        1
     112 20.4   29             76       58             31            5             0.353        0

--- 2. RATIONAL AGENT (PEAS) ---
  Performance : Accuracy, Sensitivity, F1-Score
  Environment : Patient records, lab results
  Actuators   : Diagnosis, Risk Level, Confidence
